# Phase 4 — Audio Analysis (Speech Biomarker Extraction)

**Multimodal Autism Screening System — Phase 4 of 6**

| Item | Value |
|---|---|
| Input | Audio recording (`.mp3`, `.wav`, `.m4a`) or video (`.mp4`) |
| Tools | Whisper (transcription) · Silero-VAD (voice activity) · OpenSMILE (acoustics) |
| Training | None — pre-trained models only |
| Output | `audio_features.csv` (one feature row per recording, ready for Phase 6 fusion) |

**Extracted features:** Speech Rate · Pause Count · Pause Duration (avg/total/max) · Response Latency · Pitch (mean/std) · Energy/Loudness (mean/std)

> ⚠️ This is a screening and decision-support aid. Outputs are risk indicators, not a clinical diagnosis.

## 1 · Install dependencies
Takes ~2–4 minutes on a fresh Colab runtime. GPU runtime is recommended (Runtime → Change runtime type → T4) for faster Whisper transcription, but CPU also works.

In [ ]:
!apt-get install -y -qq ffmpeg
# Removed pandas and numpy from the upgrade list to prevent dependency conflicts
!pip install -q -U openai-whisper silero-vad opensmile soundfile

import torch, whisper, opensmile
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("whisper:", whisper.__version__)
print("opensmile:", opensmile.__version__)

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


torch: 2.11.0+cu130 | CUDA available: True
whisper: 20250625
opensmile: 2.6.0


## 2 · Configuration
Edit these settings as needed.

- **`PAUSE_MIN_S`** — minimum silence gap between speech segments that counts as a *pause* (0.3 s is a common threshold).
- **`PROMPT_END_TIME_S`** — set this to the timestamp (in seconds) when the examiner's question/prompt ends if your recording includes the question. Response Latency is then measured from that point. Leave it as `None` to measure latency from the start of the recording.
- **`WHISPER_MODEL`** — `tiny`/`base` are fast, `small` is more accurate, `medium` is best but slow on CPU.

In [ ]:
# ---------------- Configuration ----------------
WHISPER_MODEL   = "base"      # tiny | base | small | medium
PAUSE_MIN_S     = 0.30        # min gap (s) between speech segments counted as a pause
PROMPT_END_TIME_S = None      # e.g. 5.0 if the question ends at 5 s; None = measure from file start
VAD_THRESHOLD   = 0.5         # Silero-VAD speech probability threshold
OUTPUT_CSV      = "audio_features.csv"
# ------------------------------------------------

## 3 · Upload / locate the input file
Run the cell and use the upload button, **or** set `INPUT_PATH` to a file already on the runtime (e.g. from Google Drive). Accepts `.mp3`, `.wav`, `.m4a`, `.mp4`, `.mov`, `.mkv`.

In [ ]:
import os
from google.colab import files  # remove this line if running outside Colab

INPUT_PATH = None  # e.g. "/content/drive/MyDrive/recordings/child_01.mp4"

if INPUT_PATH is None:
    uploaded = files.upload()
    INPUT_PATH = next(iter(uploaded))

assert os.path.exists(INPUT_PATH), f"File not found: {INPUT_PATH}"
print("Input file:", INPUT_PATH, f"({os.path.getsize(INPUT_PATH)/1e6:.2f} MB)")

Saving asd01.mp3 to asd01.mp3
Input file: asd01.mp3 (2.20 MB)


## 4 · Convert input to 16 kHz mono WAV
`ffmpeg` extracts the audio track from MP4 (or re-encodes MP3) into the format expected by Whisper, Silero-VAD and OpenSMILE.

In [ ]:
import subprocess, os

def to_wav_16k(input_path, out_path="audio_16k.wav"):
    """Convert any audio/video container to 16 kHz mono PCM WAV."""
    cmd = ["ffmpeg", "-y", "-i", input_path, "-vn",
           "-ac", "1", "-ar", "16000", "-acodec", "pcm_s16le", out_path]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0 or not os.path.exists(out_path):
        raise RuntimeError("ffmpeg failed:\n" + r.stderr[-2000:])
    return out_path

WAV_PATH = to_wav_16k(INPUT_PATH)

import soundfile as sf
info = sf.info(WAV_PATH)
DURATION_S = info.duration
print(f"Converted → {WAV_PATH} | {DURATION_S:.2f} s | {info.samplerate} Hz mono")

Converted → audio_16k.wav | 137.69 s | 16000 Hz mono


## 5 · Voice Activity Detection → Pause features & Response Latency
Silero-VAD finds every speech segment. Gaps between consecutive segments longer than `PAUSE_MIN_S` are counted as pauses. **Response Latency** = time from the start of the recording (or `PROMPT_END_TIME_S`) to the first detected speech.

In [ ]:
import numpy as np
from silero_vad import load_silero_vad, get_speech_timestamps

def load_wav_tensor(path):
    """Load WAV as a 1-D float32 torch tensor (avoids the torchaudio dependency)."""
    data, sr = sf.read(path, dtype="float32")
    assert sr == 16000
    return torch.from_numpy(data)

vad_model = load_silero_vad()
wav = load_wav_tensor(WAV_PATH)

speech_ts = get_speech_timestamps(
    wav, vad_model, sampling_rate=16000,
    threshold=VAD_THRESHOLD,
    min_speech_duration_ms=150,      # ignore very short blips
    min_silence_duration_ms=int(PAUSE_MIN_S * 1000),
)

# segments in seconds
segments = [(t["start"] / 16000.0, t["end"] / 16000.0) for t in speech_ts]
speech_time = sum(e - s for s, e in segments)

# pauses = gaps between consecutive speech segments
pauses = [segments[i+1][0] - segments[i][1] for i in range(len(segments) - 1)]
pauses = [p for p in pauses if p >= PAUSE_MIN_S]

pause_count        = len(pauses)
total_pause_dur    = float(np.sum(pauses)) if pauses else 0.0
avg_pause_dur      = float(np.mean(pauses)) if pauses else 0.0
max_pause_dur      = float(np.max(pauses)) if pauses else 0.0

# response latency = first speech onset relative to reference point
ref = PROMPT_END_TIME_S if PROMPT_END_TIME_S is not None else 0.0
response_latency = max(0.0, segments[0][0] - ref) if segments else float("nan")

speech_ratio = speech_time / DURATION_S if DURATION_S > 0 else 0.0

print(f"Speech segments detected : {len(segments)}")
print(f"Total speaking time      : {speech_time:.2f} s  ({speech_ratio:.0%} of recording)")
print(f"Pause count              : {pause_count}")
print(f"Avg / total / max pause  : {avg_pause_dur:.2f} / {total_pause_dur:.2f} / {max_pause_dur:.2f} s")
print(f"Response latency         : {response_latency:.2f} s (reference: {ref:.2f} s)")
print(f"First 5 segments         : {[(round(s,2), round(e,2)) for s,e in segments[:5]]}")

Speech segments detected : 34
Total speaking time      : 63.58 s  (46% of recording)
Pause count              : 33
Avg / total / max pause  : 2.19 / 72.29 / 6.56 s
Response latency         : 1.31 s (reference: 0.00 s)
First 5 segments         : [(1.31, 3.39), (3.91, 5.12), (7.39, 9.41), (12.87, 15.01), (16.19, 18.3)]


## 6 · Whisper transcription → Speech Rate
Speech Rate = recognised words ÷ speaking time × 60 (words per minute).

In [ ]:
import whisper, re

wmodel = whisper.load_model(WHISPER_MODEL)   # uses GPU automatically if available
result = wmodel.transcribe(WAV_PATH, fp16=torch.cuda.is_available())

transcript = result["text"].strip()
detected_lang = result.get("language", "?")
words = re.findall(r"[\w'-]+", transcript)
word_count = len(words)

# words per minute of actual speaking time (fallback: whole recording)
basis = speech_time if speech_time > 0 else DURATION_S
speech_rate_wpm = round(word_count / basis * 60.0, 1) if basis > 0 else 0.0

print(f"Detected language : {detected_lang}")
print(f"Word count        : {word_count}")
print(f"Speech rate       : {speech_rate_wpm} WPM")
print(f"Transcript        : {transcript[:400]}{'...' if len(transcript) > 400 else ''}")

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 170MiB/s]


Detected language : nl
Word count        : 238
Speech rate       : 224.6 WPM
Transcript        : Een india die wij ze naar een appel en zie je een appelboel. Hij probeert ze appelst te plukken. Hij stond te besteen in vioederaaf. Dan kwam er een kouw in met een lader. Die is de lader tegen de appelboelmaen en plukte de appel. Hij gaf de appel die hij had gepleukt. De india. En die india was blij. Hij is goed. De duurplinsjes gaat naar de eifkoepbaar en wil hem eind gekopen. Ze heeft er in gek...


## 7 · OpenSMILE → Pitch & Energy
eGeMAPSv02 functionals give standardised acoustic summaries:
- **Pitch** — mean F0 (converted from semitones to Hz) and its variability (std, in semitones).
- **Energy** — mean loudness and loudness variability (OpenSMILE loudness scale ≈ perceptual energy), plus equivalent sound level.

In [ ]:
import opensmile

smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)
feats = smile.process_file(WAV_PATH)

# Pitch: F0 is given in semitones relative to 27.5 Hz → convert to Hz
f0_semitone_mean = float(feats["F0semitoneFrom27.5Hz_sma3nz_amean"].iloc[0])
f0_semitone_std  = float(feats["F0semitoneFrom27.5Hz_sma3nz_stddevNorm"].iloc[0])
pitch_mean_hz    = round(27.5 * 2 ** (f0_semitone_mean / 12.0), 2)
pitch_std_semitones = round(f0_semitone_std, 3)

# Energy: perceptual loudness (mean + variability) and equivalent sound level
energy_loudness_mean = round(float(feats["loudness_sma3_amean"].iloc[0]), 4)
energy_loudness_std  = round(float(feats["loudness_sma3_stddevNorm"].iloc[0]), 4)
energy_dB            = round(float(feats["equivalentSoundLevel_dBp"].iloc[0]), 2)

print(f"Pitch  : {pitch_mean_hz} Hz (mean), variability {pitch_std_semitones} semitones")
print(f"Energy : loudness mean {energy_loudness_mean}, std {energy_loudness_std}, {energy_dB} dB")

Pitch  : 195.65 Hz (mean), variability 0.099 semitones
Energy : loudness mean 0.6257, std 0.4847, -29.75 dB


## 8 · Assemble feature row → `audio_features.csv`
All Phase-4 biomarkers in one row — the exact shape Phase 6 fusion expects (one row per child/recording). Running this cell repeatedly **appends** new rows, so you can batch-process many recordings.

In [ ]:
import pandas as pd
from datetime import datetime

row = {
    # identifiers / metadata
    "file": os.path.basename(INPUT_PATH),
    "duration_s": round(DURATION_S, 2),
    "processed_at": datetime.now().isoformat(timespec="seconds"),
    # ---- the six Phase-4 biomarkers ----
    "speech_rate_wpm": speech_rate_wpm,          # Speech Rate
    "pause_count": pause_count,                  # Pause Count
    "avg_pause_duration_s": round(avg_pause_dur, 3),   # Pause Duration
    "total_pause_duration_s": round(total_pause_dur, 3),
    "max_pause_duration_s": round(max_pause_dur, 3),
    "response_latency_s": round(response_latency, 3),  # Response Latency
    "pitch_mean_hz": pitch_mean_hz,              # Pitch
    "pitch_std_semitones": pitch_std_semitones,
    "energy_loudness_mean": energy_loudness_mean,      # Energy
    "energy_loudness_std": energy_loudness_std,
    "energy_db": energy_dB,
    # ---- extra context (useful for debugging / Phase 6) ----
    "speech_time_s": round(speech_time, 2),
    "speech_ratio": round(speech_ratio, 3),
    "word_count": word_count,
    "language": detected_lang,
}

df_new = pd.DataFrame([row])
if os.path.exists(OUTPUT_CSV):
    df = pd.concat([pd.read_csv(OUTPUT_CSV), df_new], ignore_index=True)
else:
    df = df_new
df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved → {OUTPUT_CSV}  ({len(df)} row(s))")
display(df.tail(3).T)

# download the CSV to your machine
from google.colab import files
files.download(OUTPUT_CSV)

Saved → audio_features.csv  (1 row(s))


,0
file,asd01.mp3
duration_s,137.69
processed_at,2026-10-30T14:24:49
speech_rate_wpm,224.6
pause_count,33
avg_pause_duration_s,2.191
total_pause_duration_s,72.292
max_pause_duration_s,6.564
response_latency_s,1.314
pitch_mean_hz,195.65


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9 · (Optional) Batch mode — process a whole folder
Point `FOLDER` at a directory of recordings (e.g. a Google Drive folder) and every `.mp3/.wav/.m4a/.mp4` inside will be processed and appended to `audio_features.csv`.

In [ ]:
import glob, os

FOLDER = None  # e.g. "/content/drive/MyDrive/recordings"

if FOLDER:
    paths = sorted(p for ext in ("*.mp3","*.wav","*.m4a","*.mp4","*.mov","*.mkv")
                   for p in glob.glob(os.path.join(FOLDER, ext)))
    print(f"Found {len(paths)} files")
    for INPUT_PATH in paths:
        print("\n===", INPUT_PATH, "===")
        try:
            WAV_PATH = to_wav_16k(INPUT_PATH)
            info = sf.info(WAV_PATH); DURATION_S = info.duration

            wav = load_wav_tensor(WAV_PATH)
            speech_ts = get_speech_timestamps(wav, vad_model, sampling_rate=16000,
                                              threshold=VAD_THRESHOLD,
                                              min_speech_duration_ms=150,
                                              min_silence_duration_ms=int(PAUSE_MIN_S*1000))
            segments = [(t["start"]/16000.0, t["end"]/16000.0) for t in speech_ts]
            speech_time = sum(e-s for s,e in segments)
            pauses = [segments[i+1][0]-segments[i][1] for i in range(len(segments)-1)]
            pauses = [p for p in pauses if p >= PAUSE_MIN_S]
            pause_count = len(pauses)
            total_pause_dur = float(np.sum(pauses)) if pauses else 0.0
            avg_pause_dur   = float(np.mean(pauses)) if pauses else 0.0
            max_pause_dur   = float(np.max(pauses)) if pauses else 0.0
            ref = PROMPT_END_TIME_S if PROMPT_END_TIME_S is not None else 0.0
            response_latency = max(0.0, segments[0][0]-ref) if segments else float("nan")
            speech_ratio = speech_time/DURATION_S if DURATION_S>0 else 0.0

            result = wmodel.transcribe(WAV_PATH, fp16=torch.cuda.is_available())
            transcript = result["text"].strip(); detected_lang = result.get("language","?")
            word_count = len(re.findall(r"[\w'-]+", transcript))
            basis = speech_time if speech_time>0 else DURATION_S
            speech_rate_wpm = round(word_count/basis*60.0, 1) if basis>0 else 0.0

            feats = smile.process_file(WAV_PATH)
            f0m = float(feats["F0semitoneFrom27.5Hz_sma3nz_amean"].iloc[0])
            pitch_mean_hz = round(27.5*2**(f0m/12.0), 2)
            pitch_std_semitones = round(float(feats["F0semitoneFrom27.5Hz_sma3nz_stddevNorm"].iloc[0]),3)
            energy_loudness_mean = round(float(feats["loudness_sma3_amean"].iloc[0]),4)
            energy_loudness_std  = round(float(feats["loudness_sma3_stddevNorm"].iloc[0]),4)
            energy_dB = round(float(feats["equivalentSoundLevel_dBp"].iloc[0]),2)

            row = {"file": os.path.basename(INPUT_PATH), "duration_s": round(DURATION_S,2),
                   "processed_at": datetime.now().isoformat(timespec="seconds"),
                   "speech_rate_wpm": speech_rate_wpm, "pause_count": pause_count,
                   "avg_pause_duration_s": round(avg_pause_dur,3),
                   "total_pause_duration_s": round(total_pause_dur,3),
                   "max_pause_duration_s": round(max_pause_dur,3),
                   "response_latency_s": round(response_latency,3),
                   "pitch_mean_hz": pitch_mean_hz, "pitch_std_semitones": pitch_std_semitones,
                   "energy_loudness_mean": energy_loudness_mean,
                   "energy_loudness_std": energy_loudness_std, "energy_db": energy_dB,
                   "speech_time_s": round(speech_time,2), "speech_ratio": round(speech_ratio,3),
                   "word_count": word_count, "language": detected_lang}
            df = pd.concat([pd.read_csv(OUTPUT_CSV) if os.path.exists(OUTPUT_CSV) else pd.DataFrame(),
                            pd.DataFrame([row])], ignore_index=True)
            df.to_csv(OUTPUT_CSV, index=False)
            print("  ✓ appended")
        except Exception as ex:
            print("  ✗ FAILED:", ex)
    print("\nDone. Final table:")
    display(pd.read_csv(OUTPUT_CSV))
else:
    print("Set FOLDER to a directory path to enable batch mode.")

---
## Notes for the report / Phase 6 integration

| Column | Maps to report feature | Unit |
|---|---|---|
| `speech_rate_wpm` | Speech Rate | words/min |
| `pause_count` | Pause Count | count |
| `avg_pause_duration_s` | Pause Duration | seconds |
| `response_latency_s` | Response Latency | seconds |
| `pitch_mean_hz` | Pitch | Hz |
| `energy_loudness_mean` | Energy | OpenSMILE loudness |

**Definitions used (keep consistent across recordings!):**
- *Pause* = silence gap ≥ `PAUSE_MIN_S` (0.3 s) between two VAD speech segments.
- *Response Latency* = first speech onset minus `PROMPT_END_TIME_S` (or file start if unset).
- *Speech Rate* = Whisper word count ÷ VAD speaking time × 60.

**Troubleshooting**
- *No speech detected* → lower `VAD_THRESHOLD` to 0.3 or check the recording volume.
- *Too many/few pauses* → adjust `PAUSE_MIN_S` (0.25–0.5 s range is typical).
- *Whisper out of memory* → use a smaller `WHISPER_MODEL` (`tiny`/`base`) or shorter clips.